In [1]:
import numpy as np
import pandas as pd

In [2]:
df_customers = pd.read_csv('dataset/olist_customers_dataset.csv')
df_sellers = pd.read_csv('dataset/olist_sellers_dataset.csv')
df_products = pd.read_csv('dataset/olist_products_dataset.csv')
df_orders = pd.read_csv('dataset/olist_orders_dataset.csv')
df_order_items = pd.read_csv('dataset/olist_order_items_dataset.csv')
df_order_payments = pd.read_csv('dataset/olist_order_payments_dataset.csv')
df_product_category_name_translation = pd.read_csv('dataset/product_category_name_translation.csv')

In [8]:
DATASETS = {
    'customers': df_customers,
    'sellers': df_sellers,
    'products': df_products,
    'orders': df_orders,
    'order_items': df_order_items,
    'order_payments': df_order_payments,
    'category_translation': df_product_category_name_translation,
}

PRIMARY_KEYS = {
    'customers': ['customer_id'],
    'sellers': ['seller_id'],
    'products': ['product_id'],
    'orders': ['order_id'],
    'order_items': ['order_id', 'order_item_id'],
    'order_payments': ['order_id', 'payment_sequential'],
    'category_translation': ['product_category_name'],
}

In [10]:
DATASETS['category_translation'].head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [11]:
DATASETS['orders'].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [12]:
DATASETS['order_items'].head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [13]:
DATASETS['sellers'].head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [14]:
DATASETS['order_payments'].head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


In [15]:
from IPython.display import display


def missing_values(df):
    summary = df.isna().sum().to_frame('missing')
    summary['pct'] = (summary['missing'] / len(df) * 100).round(2)
    return summary.query('missing > 0')


def duplicate_summary(df, keys=None):
    summary = {'full_row_duplicates': df.duplicated().sum()}
    if keys:
        summary['key_duplicates'] = df.duplicated(subset=keys).sum()
        summary['unique_keys'] = df.drop_duplicates(subset=keys).shape[0]
        summary['total_rows'] = len(df)
    return pd.Series(summary)


def numeric_ranges(df):
    numeric = df.select_dtypes(include='number')
    if numeric.empty:
        return None
    return numeric.agg(['min', 'max', 'mean', 'median']).round(2).T


def categorical_overview(df, max_unique=15):
    object_cols = df.select_dtypes(include='object').columns
    if not len(object_cols):
        return None, {}

    overview = pd.DataFrame({
        'nunique': df[object_cols].nunique(dropna=False),
    })
    low_cardinality = {
        col: df[col].value_counts(dropna=False)
        for col in object_cols
        if df[col].nunique(dropna=False) <= max_unique
    }
    return overview, low_cardinality


def whitespace_issues(df):
    rows = []
    for col in df.select_dtypes(include='object').columns:
        series = df[col].dropna().astype(str)
        count = (series != series.str.strip()).sum()
        if count:
            rows.append({'column': col, 'rows_with_whitespace': count})
    return pd.DataFrame(rows)


def run_general_checks(name, df, keys=None):
    print('=' * 60)
    print(name.upper())
    print('=' * 60)
    print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n')

    print('Dtypes:')
    display(df.dtypes.to_frame('dtype'))

    missing = missing_values(df)
    print('Missing values:')
    display(missing if not missing.empty else 'None')

    print('Duplicates & key uniqueness:')
    display(duplicate_summary(df, keys).to_frame('count'))

    numeric = numeric_ranges(df)
    if numeric is not None:
        print('Numeric ranges:')
        display(numeric)

    overview, value_counts = categorical_overview(df)
    if overview is not None:
        print('Categorical columns (unique counts):')
        display(overview)
        for col, counts in value_counts.items():
            print(f'Value counts — {col}:')
            display(counts.to_frame('count'))

    whitespace = whitespace_issues(df)
    print('Whitespace issues:')
    display(whitespace if not whitespace.empty else 'None')
    print()

In [16]:
for name, df in DATASETS.items():
    run_general_checks(name, df, keys=PRIMARY_KEYS.get(name))

CUSTOMERS
Shape: 99,441 rows x 5 columns

Dtypes:


,dtype
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


Missing values:


'None'

Duplicates & key uniqueness:


...

,count
full_row_duplicates,0
key_duplicates,0
unique_keys,71
total_rows,71


Categorical columns (unique counts):


,nunique
product_category_name,71
product_category_name_english,71


Whitespace issues:


'None'